In [ ]:
import json
import os
from pathlib import Path

# This notebook is Kaggle-only.
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle").exists()
if not IN_KAGGLE:
    raise EnvironmentError(
        "This notebook is configured for Kaggle only. "
        "Run it in a Kaggle Notebook environment."
    )

# Use Kaggle working directory as project root for outputs/temp files.
project_root = Path("/kaggle/working").resolve()
print(f"Resolved project_root: {project_root}")
print(f"IN_KAGGLE={IN_KAGGLE}")

In [ ]:
KAGGLE_INPUT_ROOT = Path("/kaggle/input")

def get_jsonl_data(file_path: str):
    rows = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON at line {line_no} in {file_path}: {e}") from e
    return rows

# Explicit Kaggle dataset paths.
corpus_path = Path(
    "/kaggle/input/datasets/trinhthaison/corpus-legal-answer-question-v1/corpus.jsonl"
)
chunk_store_path = Path(
    "/kaggle/input/datasets/trinhthaison/chunk-store-legal-answer-question-v1/chunk_store.jsonl"
)

if not corpus_path.exists():
    raise FileNotFoundError(f"corpus.jsonl not found: {corpus_path}")
if not chunk_store_path.exists():
    raise FileNotFoundError(f"chunk_store.jsonl not found: {chunk_store_path}")

print(f"Using corpus_path: {corpus_path}")
print(f"Using chunk_store_path: {chunk_store_path}")

corpus = get_jsonl_data(str(corpus_path))
chunk_store = get_jsonl_data(str(chunk_store_path))

print(corpus[0])
print(chunk_store[0])

In [ ]:
from collections import defaultdict
import numpy as np
import random
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

class DualEncoder(torch.nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.q_encoder = AutoModel.from_pretrained(model_name)
        self.d_encoder = AutoModel.from_pretrained(model_name)

    @staticmethod
    def mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = torch.sum(last_hidden_state * mask, dim=1)
        denom = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / denom

    def encode_questions(self, q_tok):
        out = self.q_encoder(
            input_ids=q_tok["input_ids"],
            attention_mask=q_tok["attention_mask"],
        )
        emb = self.mean_pool(out.last_hidden_state, q_tok["attention_mask"])
        return F.normalize(emb, p=2, dim=-1)

    def encode_docs(self, d_tok):
        out = self.d_encoder(
            input_ids=d_tok["input_ids"],
            attention_mask=d_tok["attention_mask"],
        )
        emb = self.mean_pool(out.last_hidden_state, d_tok["attention_mask"])
        return F.normalize(emb, p=2, dim=-1)

In [ ]:
# Explicit Kaggle dataset paths.
CHECKPOINT_PATH = Path(
    "/kaggle/input/datasets/trinhthaison/dual-encoder-resume-v1-legal-question-answer-v1/best_dual_encoder_resume_v1.pt"
)
VECTOR_EMBEDDINGS_PATH = Path(
    "/kaggle/input/datasets/trinhthaison/vector-embeddings-v1/vector_embeddings_v1.pt"
)

def normalize_whitespace(text):
    return " ".join((text or "").split())

def apply_lora_if_needed(model: DualEncoder, cfg: dict):
    lora_r = int(cfg.get("lora_r", 0) or 0)
    if lora_r <= 0:
        return model

    if not PEFT_AVAILABLE:
        raise ImportError(
            "Checkpoint expects LoRA adapters but `peft` is not installed. "
            "Install with: pip install peft (or pip install -r requirements.txt)."
        )

    lora_cfg = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=lora_r,
        lora_alpha=int(cfg.get("lora_alpha", 32)),
        lora_dropout=float(cfg.get("lora_dropout", 0.1)),
        target_modules=cfg.get("lora_target_modules", ["query", "key", "value"]),
        bias="none",
    )
    model.q_encoder = get_peft_model(model.q_encoder, lora_cfg)
    model.d_encoder = get_peft_model(model.d_encoder, lora_cfg)
    return model

def resolve_checkpoint_path() -> Path:
    p = CHECKPOINT_PATH
    if not p.exists():
        raise FileNotFoundError(f"Checkpoint path does not exist: {p}")
    return p

def resolve_vector_embeddings_path() -> Path:
    p = VECTOR_EMBEDDINGS_PATH
    if not p.exists():
        raise FileNotFoundError(f"Vector embeddings path does not exist: {p}")
    return p

def load_checkpoint_compat(path: Path):
    # Works across torch versions where `weights_only` may or may not exist.
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def load_vector_tensor(path: Path) -> torch.Tensor:
    try:
        vec = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        vec = torch.load(path, map_location="cpu")

    if not isinstance(vec, torch.Tensor):
        raise TypeError(f"Expected a Tensor in {path}, got {type(vec)}")
    if vec.ndim != 2:
        raise ValueError(f"Expected 2D embeddings tensor, got shape={tuple(vec.shape)}")
    if vec.dtype != torch.float32:
        vec = vec.float()
    return vec.contiguous().cpu()

In [ ]:
try:
    from peft import LoraConfig, TaskType, get_peft_model
    PEFT_AVAILABLE = True
except Exception:
    PEFT_AVAILABLE = False

top_k = 10
top_k_negs = 5
progress_every_rows = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = resolve_checkpoint_path()
vector_embeddings_path = resolve_vector_embeddings_path()

checkpoint = load_checkpoint_compat(checkpoint_path)
precomputed_embeddings = load_vector_tensor(vector_embeddings_path)

cfg = checkpoint.get("config", {})
model_name = checkpoint.get("model_name") or cfg.get("model_name") or "vinai/phobert-base"
max_q_len = int(cfg.get("max_q_len", 96))
max_c_len = int(cfg.get("max_c_len", 256))

print(f"Loading checkpoint: {checkpoint_path}")
print(f"Using vector_embeddings_path: {vector_embeddings_path}")
print(f"Embedding shape: {tuple(precomputed_embeddings.shape)}")
print(f"Model: {model_name} | q_len={max_q_len} | c_len={max_c_len}")

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = DualEncoder(model_name)
model = apply_lora_if_needed(model, cfg)

state_dict = checkpoint.get("model_state_dict", checkpoint)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print(f"Missing keys while loading: {len(missing)}")
if unexpected:
    print(f"Unexpected keys while loading: {len(unexpected)}")

model = model.to(device)
model.eval()

In [ ]:
# Pre-index chunks by doc_id and keep mapping to precomputed embedding row indices.
chunks_by_doc = defaultdict(list)
embedding_row = 0
for chunk in chunk_store:
    doc_id = chunk.get("doc_id")
    article_id = chunk.get("article_id")
    chunk_text = normalize_whitespace(chunk.get("text"))

    if not chunk_text:
        continue

    if embedding_row >= precomputed_embeddings.shape[0]:
        raise ValueError(
            "Embedding rows are fewer than non-empty chunk rows. "
            "Please ensure vector_embeddings_v1.pt was built from the same chunk_store.jsonl."
        )

    if doc_id is not None:
        chunks_by_doc[doc_id].append(
            {
                "article_id": article_id,
                "text": chunk_text,
                "emb_idx": embedding_row,
            }
        )

    embedding_row += 1

if embedding_row != precomputed_embeddings.shape[0]:
    raise ValueError(
        f"Embedding/chunk mismatch: non-empty chunks={embedding_row}, "
        f"embedding_rows={precomputed_embeddings.shape[0]}. "
        "Please use matching artifacts."
    )

list_of_doc_ids = list(set(row.get("doc_id") for row in corpus))

In [ ]:
training_data = []
total_rows = len(corpus)
rows_with_chunks = 0
qa_seen = 0
qa_valid = 0

print(f"Starting retrieval with precomputed chunk embeddings for {total_rows} corpus rows...")

for row_idx, row in enumerate(corpus, start=1):
    doc_id = row.get("doc_id")
    article_id = row.get("article_id")
    chunk_items = chunks_by_doc.get(doc_id, [])

    if not chunk_items:
        if row_idx % progress_every_rows == 0 or row_idx == total_rows:
            print(
                f"[Progress] rows={row_idx}/{total_rows} | rows_with_chunks={rows_with_chunks} | "
                f"qa_seen={qa_seen} | qa_valid={qa_valid} | samples={len(training_data)}"
            )
        continue

    rows_with_chunks += 1
    qa_pairs = row.get("generated_qa_pairs", [])
    # Get question which has medium difficulty (index 1)
    easy_qa_pair = qa_pairs[1] if qa_pairs else None

    if easy_qa_pair:
        qa_seen += 1
        question = normalize_whitespace(easy_qa_pair.get("question"))
        answer = normalize_whitespace(easy_qa_pair.get("answer"))

        if not question or not answer:
            continue

        qa_valid += 1

        with torch.no_grad():
            q_tok = tokenizer(
                [question],
                padding=True,
                truncation=True,
                max_length=max_q_len,
                return_tensors="pt",
            )
            q_tok = {k: v.to(device) for k, v in q_tok.items()}
            q_emb = model.encode_questions(q_tok).cpu().squeeze(0)

        emb_indices = [c["emb_idx"] for c in chunk_items]
        d_emb = precomputed_embeddings[emb_indices]
        if d_emb.numel() == 0:
            continue

        scores = torch.matmul(d_emb, q_emb)
        k = min(top_k, scores.shape[0])
        if k <= 0:
            continue

        top_idx = torch.topk(scores, k=k, largest=True).indices.tolist()
        top_chunks = [chunk_items[i] for i in top_idx]

        pos_candidate = [c["text"] for c in top_chunks if c.get("article_id") == article_id][:1]
        neg_candidates = [c["text"] for c in top_chunks if c.get("article_id") != article_id][:top_k_negs]

        # If negatives are insufficient, fill from other chunks in same doc_id ranked by dense score.
        neg_cand_len = len(neg_candidates)
        if neg_cand_len < top_k_negs:
            used_texts = set(pos_candidate) | set(neg_candidates)
            ranked_all_idx = torch.argsort(scores, descending=True).tolist()

            need = top_k_negs - neg_cand_len
            for idx_all in ranked_all_idx:
                cand_article_id = chunk_items[idx_all].get("article_id")
                cand_text = chunk_items[idx_all]["text"]
                if cand_article_id == article_id:
                    continue
                if cand_text in used_texts:
                    continue
                neg_candidates.append(cand_text)
                used_texts.add(cand_text)
                if len(neg_candidates) >= top_k_negs or need <= 0:
                    break

        # If negatives are insufficient, fill from other chunks in other doc_ids randomly.
        neg_cand_len = len(neg_candidates)
        need = top_k_negs - neg_cand_len
        used_texts = set(pos_candidate) | set(neg_candidates)
        other_doc_ids = [other_doc_id for other_doc_id in list_of_doc_ids if other_doc_id != doc_id]
        selected_doc_ids = set()
        while need > 0:
            random_doc_id = None
            while True:
                random_doc_id = random.choice(other_doc_ids)
                if random_doc_id not in selected_doc_ids:
                    selected_doc_ids.add(random_doc_id)
                    break

            provider_chunks = chunks_by_doc.get(random_doc_id, [])

            for c in provider_chunks:
                cand_text = c["text"]
                if cand_text in used_texts:
                    continue
                neg_candidates.append(cand_text)
                used_texts.add(cand_text)
                need -= 1
                if need == 0:
                    break

        sample = {
            "doc_id": doc_id,
            "article_id": article_id,
            "question": question,
            "answer": answer,
            "positive": pos_candidate,
            "negative_chunks": neg_candidates,
        }
        training_data.append(sample)

    if row_idx % progress_every_rows == 0 or row_idx == total_rows:
        print(
            f"[Progress] rows={row_idx}/{total_rows} | rows_with_chunks={rows_with_chunks} | "
            f"qa_seen={qa_seen} | qa_valid={qa_valid} | samples={len(training_data)}"
        )

print(
    f"Done. rows={total_rows}, rows_with_chunks={rows_with_chunks}, "
    f"qa_seen={qa_seen}, qa_valid={qa_valid}, samples={len(training_data)}"
)

In [ ]:
print("Number of training samples (version 2):", len(training_data))
if training_data:
    print(training_data[0])
else:
    print("No training samples were created. Check corpus/chunk quality and filtering conditions.")

In [ ]:
# Add more 20% data from training data version 1
training_data_v1_path = Path(
    "/kaggle/input/datasets/trinhthaison/training-data-v1-legal-question-answer-v1/training_data_v1.jsonl"
)

def _load_get_jsonl_data_func():
    # Preferred import when repo layout is available as a Python package path.
    try:
        from utils.get_jsonl_data import get_jsonl_data as fn
        return fn
    except Exception:
        pass

    # Fallback to direct file load for Kaggle dataset mounts.
    candidate_files = [
        ROOT / "utils" / "get_jsonl_data.py",
        Path("/kaggle/input/datasets/trinhthaison/legal-question-answer-v1/utils/get_jsonl_data.py"),
    ]
    for py_file in candidate_files:
        if py_file.exists():
            spec = importlib.util.spec_from_file_location("get_jsonl_data_dynamic", str(py_file))
            if spec is not None and spec.loader is not None:
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)
                return module.get_jsonl_data

    raise ModuleNotFoundError(
        "Cannot load get_jsonl_data. Ensure utils/get_jsonl_data.py exists in repo or Kaggle input dataset."
    )

get_jsonl_data = _load_get_jsonl_data_func()
training_data_v1 = get_jsonl_data(str(training_data_v1_path))

if not training_data_v1:
    print(f"Warning: No data found in {training_data_v1_path}. Skipping merge.")
else:
    print(f"Loaded {len(training_data_v1)} samples from training_data_v1.")
    # Shuffle and take 20% of v1 data
    np.random.shuffle(training_data_v1)
    subset_size = max(1, len(training_data_v1) // 5)
    training_data_v1_subset = training_data_v1[:subset_size]
    print(f"Adding {len(training_data_v1_subset)} samples from training_data_v1 to current training data.")
    training_data.extend(training_data_v1_subset)

In [ ]:
print("Number of training samples (version 2) after merging :", len(training_data))

In [ ]:
# Save the training data to a JSONL file in Kaggle working directory
import json
from pathlib import Path

output_path = Path("/kaggle/working/training_data_v2.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    for sample in training_data:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f"Saved to: {output_path}")